# Session 17 lab: forests, boosting and overfitting

You will compare four models on the Nile Microfinance loans, watch overfitting happen, tune a forest with cross-validation, and decide which model to ship.

Rhythm for every step: **PREDICT** first, **run** the cell, **READ** the output, then **CHANGE** one thing. Cells that say `None  # your code here` are yours to fill in. After each answer a `check` line tells you whether your number is right. The slides gave the ideas, so use them. Forests use random numbers, so a few checks accept a range.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# A helper so that a check never crashes while your answer is still None. You do not need to read it.
import numpy as np
import pandas as pd

def safe(f):
    try:
        return float(f())
    except Exception:
        return float("nan")

---
## Step 1: Prepare the loans and split them

The same preparation as Sessions 14 to 16. The two impossible ages (0 and 199) become the median age, so all 8,000 loans stay. Blank incomes get a flag and the median income. We add instalment_to_income (the instalment-to-income ratio) and leave out `collection_calls` because it leaks the answer (Session 16). The median income is computed on all 8,000 loans before the split, on purpose: it is simple and the leak is tiny. Session 20 shows the leak-free way.

**PREDICT first.** How many of the 8,000 loans do you expect to keep? (Hint: the data dictionary mentions two impossible ages. We replace them, we do not delete the rows.)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans = pd.read_csv("loans.csv")
bad_age = (loans["age"] < 18) | (loans["age"] > 100)   # two impossible ages: 0 and 199
loans.loc[bad_age, "age"] = loans.loc[~bad_age, "age"].median()   # the median of the valid ages (36) keeps all 8,000 rows
loans["income_missing"] = loans["monthly_income_ugx"].isna().astype(int)   # a flag: 1 when the income was blank
loans["monthly_income_ugx"] = loans["monthly_income_ugx"].fillna(loans["monthly_income_ugx"].median())   # median imputation (642,600)
# instalment_to_income: loan x 1.15 / term / monthly income. The 1.15 (loan plus 15 percent interest and fees) is an ILLUSTRATIVE assumption: it is not in the data.
loans["instalment_to_income"] = loans["loan_amount_ugx"] * 1.15 / loans["term_months"] / loans["monthly_income_ugx"]
print(len(loans), "loans kept")

In [ ]:
features = None   # your code here: a list of the 12 column names (the ten honest number columns, income_missing and instalment_to_income). Leave out collection_calls and default
X = None   # your code here: the 12 columns of loans
y = None   # your code here: the default column

In [ ]:
try:
    print(X.shape, round(y.mean(), 4))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = None, None, None, None   # your code here: train_test_split with test_size=0.25, stratify=y and random_state=42

In [ ]:
try:
    print(len(X_train), len(X_test), "defaults in the test rows:", y_test.sum())
    print(X_test.index[:10].tolist())   # the same rows as every Session 14 to 17 notebook
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("loans kept", safe(lambda: len(loans)), 8000, 0)
check("12 variables", safe(lambda: X.shape[1]), 12, 0)
check("default share", safe(lambda: y.mean()), 0.1255, 0.001)
check("training rows", safe(lambda: len(X_train)), 6000, 0)
check("defaults in the test rows", safe(lambda: y_test.sum()), 251, 0)
check("same test rows as the course (sum of the row numbers)", safe(lambda: sum(X_test.index)), 8081998, 0)

**READ the output.** The default share is 12.55 percent. If a model said "no default" for every loan, what accuracy would it get on the test rows (1,749 of 2,000 = 0.8745)? Why is accuracy a poor score here (Session 15)?

> My answer: 

**CHANGE one thing.** Remove `stratify=y` from train_test_split and run Steps 1 and 2 again. How many defaults are in the test rows now (about 220)? Which later numbers move, and by how much?

> What I saw: 

---
## Step 2: A simple baseline: scaled logistic regression

Scale the variables so their coefficients are comparable (standardisation), then fit a logistic regression. The default C = 1.0 (L2 penalty) shrinks the coefficients, so they depend on the scaling. We score it on the training rows and on the test rows with AUC.

**PREDICT first.** Will the AUC be higher on the training rows or on the test rows? About how big will the gap be for a model this simple?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logistic_model = None   # your code here: make_pipeline of StandardScaler() and LogisticRegression(max_iter=1000), then fit on the training rows
auc_lr_train = None   # your code here: roc_auc_score of y_train and the predicted chance of default (predict_proba, column 1)
auc_lr_test = None   # your code here: the same on the test rows

In [ ]:
try:
    print(round(auc_lr_train, 3), round(auc_lr_test, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
coefficients = None   # your code here: a Series of the 12 coefficients (logistic_model.named_steps["logisticregression"].coef_[0]) with the feature names as the index, largest first

In [ ]:
try:
    print(coefficients.round(3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("logistic AUC, training rows", safe(lambda: auc_lr_train), 0.7607, 0.005)
check("logistic AUC, test rows", safe(lambda: auc_lr_test), 0.7553, 0.005)
check("instalment_to_income coefficient", safe(lambda: coefficients["instalment_to_income"]), 0.8040, 0.03)

**READ the output.** Which variable has the largest coefficient, and does a positive sign raise or lower the risk of default? Say it in one sentence, as to a credit committee.

> My answer: 

**CHANGE one thing.** Change C in LogisticRegression to 0.01 (a stronger penalty). What happens to the coefficients and to the test AUC?

> What I saw: 

---
## Step 3: Watch a decision tree overfit

max_depth is the most yes or no questions a loan can be asked. We grow trees of eight depths and score each one on the training rows and on the test rows.

**PREDICT first.** What training AUC will a tree with no depth limit get? And what will its test AUC be: close to 0.75, or much lower?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [1, 2, 3, 4, 6, 8, 12, None]
tree_train = []
tree_test = []
for d in depths:
    tree_model = None   # your code here: a DecisionTreeClassifier with max_depth=d and random_state=42, fitted on the training rows
    tree_train.append(None)   # your code here: AUC on the training rows
    tree_test.append(None)   # your code here: AUC on the test rows

In [ ]:
try:
    tree_results = pd.DataFrame({"max_depth": [str(d) for d in depths], "train_auc": tree_train, "test_auc": tree_test}).round(3)
    print(tree_results)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
best_depth = None   # your code here: the max_depth with the highest test AUC (np.argmax of tree_test)

In [ ]:
try:
    print("depth with the best test AUC:", best_depth)
    
    import matplotlib.pyplot as plt
    plt.plot(range(len(depths)), tree_train, marker="o", label="training")
    plt.plot(range(len(depths)), tree_test, marker="o", label="test")
    plt.xticks(range(len(depths)), [str(d) for d in depths])
    plt.xlabel("max_depth")
    plt.ylabel("AUC")
    plt.legend()
    plt.show()
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("depth 3 test AUC", safe(lambda: tree_test[2]), 0.7220, 0.03)
check("no limit: training AUC", safe(lambda: tree_train[-1]), 1.0, 0.002)
check("no limit: test AUC (a range)", safe(lambda: tree_test[-1]), 0.5826, 0.05)
check("best depth is small (2 to 4)", safe(lambda: best_depth in [2, 3, 4]), 1, 0)

**READ the output.** Where does the gap between the two lines start to open? Name that gap. What is the tree doing to the training loans when the training AUC reaches 1.0?

> My answer: 

**CHANGE one thing.** Change the scoring from AUC to accuracy (use tree_model.predict and accuracy_score). Does the overfitting still show?

> What I saw: 

---
## Step 4: Two ensembles: a random forest and gradient boosting

A random forest averages many deep trees grown on bootstrap samples. Gradient boosting adds shallow trees one at a time, each fitting the residual errors so far.

**PREDICT first.** Will the forest's training AUC be near 0.75, near 0.85 or near 1.0? Will its test AUC beat the logistic regression?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = None   # your code here: a RandomForestClassifier with n_estimators=100, random_state=42, n_jobs=-1, fitted on the training rows
auc_f_train = None   # your code here: AUC on the training rows
auc_f_test = None   # your code here: AUC on the test rows

In [ ]:
try:
    print(round(auc_f_train, 3), round(auc_f_test, 3), "gap", round(auc_f_train - auc_f_test, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

**PREDICT first.** For gradient boosting with learning rate 0.1 and trees of depth 3: will the test AUC keep rising as you add trees up to 500? Sketch the two curves before you run the cell.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

boosting_model = None   # your code here: GradientBoostingClassifier(n_estimators=500, learning_rate=0.1, max_depth=3, random_state=42), fitted on the training rows
# staged_predict_proba gives the predicted chances after 1 tree, 2 trees, ... 500 trees
boost_train = [None] * 500   # your code here: a list of AUC values on the training rows, one per stage (loop over boosting_model.staged_predict_proba(X_train))
boost_test = [None] * 500   # your code here: the same on the test rows
best_n = None   # your code here: the number of trees (1 to 500) with the highest test AUC

In [ ]:
try:
    print("100 trees:", round(boost_train[99], 3), round(boost_test[99], 3))
    print("best number of trees:", best_n, round(max(boost_test), 3), "(picked from the test curve: for illustration only)")
    
    plt.plot(range(1, 501), boost_train, label="training")
    plt.plot(range(1, 501), boost_test, label="test")
    plt.xlabel("number of trees")
    plt.ylabel("AUC")
    plt.legend()
    plt.show()
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("forest training AUC", safe(lambda: auc_f_train), 1.0, 0.01)
check("forest test AUC (a range)", safe(lambda: auc_f_test), 0.7186, 0.03)
check("forest gap above 0.2", safe(lambda: auc_f_train - auc_f_test > 0.2), 1, 0)
check("boosting, 100 trees: training AUC", safe(lambda: boost_train[99]), 0.8329, 0.015)
check("boosting, 100 trees: test AUC", safe(lambda: boost_test[99]), 0.7399, 0.015)
check("boosting: best test AUC", safe(lambda: max(boost_test)), 0.7474, 0.015)
check("boosting peaks before 250 trees", safe(lambda: best_n < 250), 1, 0)

**READ the output.** How big is the forest's generalisation gap? Does the forest beat the logistic regression on the test rows? In boosting, what happens to the training and test curves after the peak?

> My answer: 

**CHANGE one thing.** Re-fit the forest with max_features=None (plain bagging). Does the test AUC go up or down? Why might that be?

> What I saw: 

---
## Step 5: Tune the forest with cross-validation, then ask what it relies on

GridSearchCV tries every combination of settings with 5-fold cross-validation on the training rows (1,200 loans in each fold). The test rows stay untouched until the end.

**PREDICT first.** The grid has 3 values of max_depth and 3 of min_samples_leaf. How many model fits will GridSearchCV run with 5 folds? Will the best cross-validated AUC be higher than the default forest's cross-validated AUC of about 0.72?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"max_depth": [3, 6, None], "min_samples_leaf": [1, 10, 30]}
search = None   # your code here: GridSearchCV of RandomForestClassifier(n_estimators=100, random_state=42) with param_grid, cv=5, scoring="roc_auc", n_jobs=-1; then fit on the training rows (a few minutes: run it once)
tuned_forest = None   # your code here: the best model, search.best_estimator_
auc_t_train = None   # your code here: its AUC on the training rows
auc_t_test = None   # your code here: its AUC on the test rows

In [ ]:
try:
    print("best settings:", search.best_params_)
    print("best mean cross-validated AUC (the best of nine cells, so a little optimistic):", round(search.best_score_, 3))
    print("tuned forest: training", round(auc_t_train, 3), "test", round(auc_t_test, 3))
    print("Several cells score within 0.005 of the best. The best value is at the edge of the grid (min_samples_leaf 30), so a real project would also try 50 and 100.")
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("best cross-validated AUC (a range)", safe(lambda: search.best_score_), 0.7461, 0.02)
check("tuned forest test AUC (a range)", safe(lambda: auc_t_test), 0.7485, 0.03)
check("tuned forest gap is smaller than the default forest gap", safe(lambda: (auc_t_train - auc_t_test) < (auc_f_train - auc_f_test)), 1, 0)

**PREDICT first.** Permutation importance shuffles one variable at a time on the test rows. Which variable do you expect to lose the most AUC? Which will lose about nothing?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.inspection import permutation_importance

imp = None   # your code here: permutation_importance of tuned_forest on the test rows, scoring="roc_auc", n_repeats=10, random_state=42
importance = None   # your code here: a Series of imp.importances_mean with the feature names as the index, largest first

In [ ]:
try:
    print(importance.round(3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("most important variable is instalment_to_income", safe(lambda: importance.index[0] == "instalment_to_income"), 1, 0)
check("instalment_to_income: drop in AUC (a range)", safe(lambda: importance.iloc[0]), 0.0609, 0.03)

**READ the output.** Which variables does the tuned forest rely on? Is the order the same as the order of the logistic coefficients in Step 2? State one thing permutation importance cannot tell you.

> My answer: 

**CHANGE one thing.** Add a copy of instalment_to_income to X_train and X_test (X_train["ratio_copy"] = X_train["instalment_to_income"]), refit the tuned forest, and run permutation importance again. What happens to the importance of instalment_to_income?

> What I saw: 

---
## Step 6: Build the comparison table and decide

Four models, three scores each: training AUC, 5-fold cross-validated AUC on the training rows, and test AUC. Then the decision. The test rows are scored once.

**PREDICT first.** Which model will have the highest test AUC? Which will have the largest gap between training and test?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import cross_val_score

cv_lr = None   # your code here: cross_val_score of logistic_model on the training rows, cv=5, scoring="roc_auc"
cv_tree = None   # your code here: the same for DecisionTreeClassifier(max_depth=3, random_state=42)
cv_boost = None   # your code here: the same for GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

results = None   # your code here: a DataFrame with the columns model, train_auc, cv_auc, test_auc for the four models (logistic regression, tree depth 3, tuned forest, boosting with 100 trees; boost_train[99] and boost_test[99]), plus gap = train_auc - test_auc, rounded to 3 places. Then results.to_csv("model_comparison.csv", index=False)

In [ ]:
try:
    print(results)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("cross-validated AUC, logistic", safe(lambda: cv_lr.mean()), 0.7528, 0.01)
check("cross-validated AUC, tree depth 3", safe(lambda: cv_tree.mean()), 0.7153, 0.03)
check("cross-validated AUC, boosting", safe(lambda: cv_boost.mean()), 0.7416, 0.015)
check("four models in the table", safe(lambda: len(results)), 4, 0)
check("simplest good model: logistic test AUC is within 0.03 of the best", safe(lambda: results["test_auc"].max() - results.loc[0, "test_auc"] < 0.03), 1, 0)

In [ ]:
show_checks()   # run this to see how many checks pass

**READ the output.** Look at the cv_auc and test_auc columns. Are the differences between the best three models bigger or smaller than the standard deviation between folds (about 0.01 for these models)?

> My answer: 

## Your decision memo

Double click this cell and write three sentences. (1) Which model would you ship to a credit committee? (2) Which two numbers from `results` support the choice? (3) What would change your mind?

> Memo: 

Save the notebook and keep `model_comparison.csv` (it is in your Colab folder: download it from the Files panel). That table and the memo are what you take away from today.